In [2]:
import numpy as np
import pandas as pd

import os
import plotly.express as px
import plotly.io as pio

import webbrowser

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

# DATA CLEANING

In [ ]:
property_df=pd.read_csv("property_data.csv")

In [ ]:
property_df.head(2)

In [ ]:
property_df.tail(2)

In [ ]:
property_df=property_df.drop_duplicates()

In [ ]:
property_df.info()

In [ ]:
property_df.isnull().sum()

In [ ]:
property_df = property_df.rename(columns={"bhk": "Bedrooms(BHK)"})
property_df = property_df.rename(columns={"builders_name": "Owner_name"})
property_df = property_df.rename(columns={"per_sqrt":"per_sqft"})
property_df = property_df.rename(columns={"car_parking":"parking_lot"})

In [ ]:
property_df['Owner_name']=property_df['Owner_name'].str.replace('Owner:','').str.replace('Agent:','')

In [ ]:
property_df['per_sqft'] = property_df['per_sqft'].str.replace('per sqft', '').str.replace('₹', '')
property_df['per_sqft'] = pd.to_numeric(property_df['per_sqft'], errors='coerce').astype('Int64')

In [ ]:
property_df['furnishing'] = property_df['furnishing'].str.replace('-', ' ')

In [ ]:
property_df['sqft'] = property_df['sqft'].str.replace('sqft', '')

In [ ]:
property_df['Bedrooms(BHK)'] = property_df['Bedrooms(BHK)'].str[:1]

In [ ]:
property_df['parking_lot'] = property_df['parking_lot'].fillna('No Parking')
property_df['no_of_Parking']=property_df['parking_lot'].str.replace('No Parking','0').str.replace('Open','').str.replace('Covered','')
property_df['no_of_Parking']=property_df['no_of_Parking'].str.replace(' ','').str.replace(',','').astype(int)

In [ ]:
def convert_price(price):
    price = str(price).replace(',', '').strip()
    if 'Cr' in price:
        return float(price.replace('Cr', '').strip()) * 10000000
    elif 'Lac' in price:
        return float(price.replace('Lac', '').strip()) * 100000
    else:
        return pd.to_numeric(price, errors='coerce')

property_df['price_of_property'] = property_df['price_of_property'].apply(convert_price)

In [ ]:
property_df['Bedrooms(BHK)'] = property_df['Bedrooms(BHK)'].replace('S', np.nan)
property_df = property_df.dropna(subset=['Bedrooms(BHK)'])
property_df['Bedrooms(BHK)'] = property_df['Bedrooms(BHK)'].astype(int)

property_df['sqft'] = property_df['sqft'].astype(int)

In [ ]:
property_df.head()

In [ ]:
property_df.dtypes

# DATA VISUALISATION

In [ ]:
html_files_path="./"
if not os.path.exists(html_files_path):
    os.makedirs(html_files_path)

In [ ]:
plot_containers=""

In [ ]:
# Save each Plotly figure to an HTML file
def save_plot_as_html(fig, filename, insight):
    global plot_containers
    filepath = os.path.join(html_files_path, filename)
    html_content = pio.to_html(fig, full_html=False, include_plotlyjs='inline')
    # Append the plot and its insight to plot_containers
    plot_containers += f"""
    <div class="plot-container" id="{filename}" onclick="openPlot('{filename}')">
        <div class="plot">{html_content}</div>
        <div class="insights">{insight}</div>
    </div>
    """
    fig.write_html(filepath, full_html=False, include_plotlyjs='inline')

In [ ]:
fig1 = px.scatter(
    property_df,
    x='sqft',
    y='price_of_property',
    color='Bedrooms(BHK)',
    title='Price vs. Sqft (Colored by Bedrooms)',
    labels={'sqft': 'Property Size (Sqft)','price_of_property':'price of property'} 
)

save_plot_as_html(
    fig1,
    "Scatter Plot 1.html",
    "This chart shows how bigger homes usually cost more. Each color is a different number of bedrooms"
)
fig1.show()

In [ ]:
# Boxplot: Price by Bedrooms (BHK)
fig2 = px.box(
    property_df,
    x='Bedrooms(BHK)',
    y='price_of_property',
    title='Price by Bedrooms (BHK)',
    labels={'price_of_property': 'Price of Property (in Lakhs)', 'Bedrooms(BHK)': 'Bedrooms (BHK)'}
)

save_plot_as_html(
    fig2,
    "Boxplot 2.html",
    "This chart compares prices for homes with different bedrooms. You can see which type costs more or less"
)
fig2.show()

In [ ]:
# furnishing_counts = property_df['furnishing'].value_counts().reset_index()
# furnishing_counts

In [ ]:
furnishing_counts = property_df['furnishing'].value_counts().reset_index()
furnishing_counts.columns = ['furnishing', 'count']

fig3 = px.bar(
    furnishing_counts,
    x='furnishing',
    y='count',
    color='furnishing',
    title='Count of Properties by Furnishing Type',
    labels={'furnishing': 'Furnishing Type', 'count': 'Number of Properties'},
    color_discrete_sequence=px.colors.sequential.OrRd,
)

save_plot_as_html(
    fig3,
    "Bar Chart 3.html",
    "This chart shows how many homes have each type of furnishing, like fully or semi furnished"
)
fig3.show()

In [ ]:
# Scatter Plot: Price per Sqft vs. Bedrooms (BHK)
fig4 = px.scatter(
    property_df,
    x='Bedrooms(BHK)',
    y='per_sqft',
    color='Bedrooms(BHK)',  # Optional: color by BHK
    title='Price per Sqft vs. Bedrooms (BHK)',
    labels={'Bedrooms(BHK)': 'Bedrooms (BHK)', 'per_sqft': 'Price per Sqft'}
)

save_plot_as_html(
    fig4,
    "Scatter Plot 4.html",
    "This chart shows if homes with more bedrooms have higher or lower price per square foot"
)
fig4.show()

In [ ]:
# Bar Chart: Count of Properties by Parking Availability
parking_counts = property_df['parking_lot'].value_counts().reset_index()
parking_counts.columns = ['parking_lot', 'count']

fig5 = px.bar(
    parking_counts,
    x='parking_lot',
    y='count',
    color='parking_lot',
    title='Count of Properties by Parking Availability',
    labels={'parking_lot': 'Parking Availability', 'count': 'Number of Properties'},
    color_discrete_sequence=['#10D91A', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A']
)

save_plot_as_html(
    fig5,
    "Bar Chart 5.html",
    "This chart shows how many homes have parking and what kind of parking they offer"
)
fig5.show() 

In [ ]:
# Correlation Heatmap of Numerical Columns
corr = property_df[['price_of_property', 'sqft', 'Bedrooms(BHK)', 'per_sqft', 'no_of_Parking']].corr()

fig6 = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale='Viridis',
    title='Correlation Heatmap of Numerical Columns',
    labels=dict(x="Features", y="Features", color="Correlation")
)

save_plot_as_html(
    fig6,
    "Correlation Heatmap 6.html",
    "This chart shows which numbers in the data are related, like price and size"
)
fig6.show()

In [ ]:
fig7 = px.histogram(
    property_df,
    x='price_of_property',
    nbins=30,
    title='Distribution of Property Prices',
    labels={'price_of_property': 'Price of Property'},
    color_discrete_sequence=['#C214D2']
)
save_plot_as_html(
    fig7,
    "Chart7.html",
    "This chart shows how many homes are in each price range"
)
fig7.show()

In [ ]:
bedroom_counts = property_df['Bedrooms(BHK)'].value_counts().reset_index()
bedroom_counts.columns = ['Bedrooms(BHK)', 'count']

fig8 = px.pie(
    bedroom_counts,
    names='Bedrooms(BHK)',
    values='count',
    title='Distribution of Properties by Number of Bedrooms',
    color_discrete_sequence=px.colors.sequential.RdBu
)
save_plot_as_html(
    fig8,
    "Chart8.html",
    "This chart shows what share of homes have 1, 2, 3, or more bedrooms"
)
fig8.show()

In [ ]:
# ...existing code...
dashboard_html = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Property Real Estate Analysis</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            background-color: #333;
            color: #fff;
            margin: 0;
            padding: 0;
        }}
        .header {{
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 20px;
            background-color: #444;
        }}
        .container {{
            display: flex;
            flex-wrap: wrap;
            justify-content: center;
            padding: 20px;
        }}
        .plot-container {{
            border: 2px solid #555;
            margin: 10px;
            padding: 10px;
            width: {plot_width}px;
            height: {plot_height}px;
            overflow: hidden;
            position: relative;
            cursor: pointer;
        }}
        .plot {{
            max-width: 100%;
            max-height: 100%;
            overflow: auto;
        }}
        .insights {{
            display: none;
            position: absolute;
            right: 10px;
            top: 10px;
            background-color: rgba(0, 0, 0, 0.7);
            padding: 5px;
            border-radius: 5px;
            color: #fff;
        }}
        .plot-container:hover .insights {{
            display: block;
        }}
    </style>
    <script>
        function openPlot(filename) {{
            window.open(filename, '_blank');
        }}
    </script>
</head>
<body>
    <div class="header">
       <a>Property Real Estate Analysis<a/>
    </div>
    <div class="container">
        {plots}
    </div>
</body>
</html>
"""
# ...existing code...

In [ ]:
final_html = dashboard_html.format(plots=plot_containers, plot_width=400, plot_height=300)
dashboard_path = os.path.join(html_files_path, "web_page.html")
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(final_html)
import webbrowser
webbrowser.open('file://' + os.path.realpath(dashboard_path))